In [ ]:
import sys, warnings
from pathlib import Path
import torch

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._forking_sequences import fork_sequences, heterogeneous_sampler

warnings.filterwarnings("ignore")
print("imports OK")

## Test — `fork_sequences` unit tests

In [ ]:
print("=" * 60)
print("TEST — fork_sequences shape assertions")
print("=" * 60)

def make_batch(B=2, T=200, C=3, Vh=1, H=6, device="cpu"):
    """Minimal collated batch as _full_series_collate_fn would produce."""
    return {
        "x_enc":          torch.randn(B, T, C, 1 + Vh, device=device),
        "available_mask": torch.ones(B, T, C, device=device),   # [B, T, C]
        "channel_mask":   torch.ones(B, C, device=device),
        "horizon":        torch.full((B,), H, dtype=torch.long, device=device),
    }

L, H, S = 32, 6, 1

# ── Training path (fcd_samples=4) ─────────────────────────────────────────────
print("\n--- Training path (fcd_samples=4) ---")
batch = make_batch(B=2, T=200, C=3, Vh=1, H=H)
out   = fork_sequences(batch, context_length=L, fcd_samples=4, horizon=H)
B_out, enc_size, C_out, feat = out["insample_y"].shape
print(f"insample_y:    {tuple(out['insample_y'].shape)}    expected [2, enc_size, 3, 2]")
print(f"outsample_y:   {tuple(out['outsample_y'].shape)}   expected [2, 4, {H}, 3]")
print(f"available_mask:{tuple(out['available_mask'].shape)}")
assert out["outsample_y"].shape == (2, 4, H, 3), \
    f"outsample_y shape wrong: {out['outsample_y'].shape}"
assert out["available_mask"].shape == (2, enc_size, 3), \
    f"available_mask shape wrong: {out['available_mask'].shape}"
print("training path shapes ✓")

# ── Val/test path (fcd_samples=-1) ────────────────────────────────────────────
print("\n--- Val/test path (fcd_samples=-1) ---")
T = 200
batch_val  = make_batch(B=2, T=T, C=3, Vh=0, H=H)
out_val    = fork_sequences(batch_val, context_length=L, fcd_samples=-1, horizon=H)
n_expected = T - L - H + 1
print(f"n_valid_fcds({T}, L={L}, H={H}, S={S}) = {n_expected}")
print(f"outsample_y:   {tuple(out_val['outsample_y'].shape)}  expected [2, {n_expected}, {H}, 3]")
assert out_val["outsample_y"].shape[1] == n_expected, \
    f"n_fcds wrong: {out_val['outsample_y'].shape[1]} != {n_expected}"
print("val/test path shapes ✓")

# ── Left-padding: windows never start in padding ───────────────────────────────
print("\n--- Left-padding: sampler skips padded timesteps ---")
T_real, T_pad = 100, 50
T_total       = T_real + T_pad
mask = torch.zeros(1, T_total, 3)       # [B, T, C]
mask[:, T_pad:, :] = 1.0               # real data starts at T_pad
batch_pad = {
    "x_enc":          torch.randn(1, T_total, 3, 1),
    "available_mask": mask,
    "horizon":        torch.tensor([H]),
}

for _ in range(20):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=4, horizon=H)
    assert ws[0].item() >= T_pad, \
        f"window_start {ws[0].item()} < T_pad {T_pad} — sampled in padding!"
print("all 20 sampled window_starts ≥ T_pad ✓")

# ── available_mask shape assertion ────────────────────────────────────────────
print("\n--- available_mask shape mismatch raises AssertionError ---")
bad_mask_batch = {
    "x_enc":          torch.randn(2, 100, 3, 1),
    "available_mask": torch.ones(2, 3, 100),   # wrong: [B, C, T] instead of [B, T, C]
    "horizon":        torch.tensor([H, H]),
}
try:
    fork_sequences(bad_mask_batch, context_length=L, fcd_samples=4, horizon=H)
    assert False, "should have raised"
except AssertionError as e:
    print(f"raised AssertionError as expected: {e}")

print("\n✓ TEST PASSED")

## Test: window_sampling start timestamp must never exceeds new max_start

In [ ]:
print("=" * 60)
print("TEST —  window_start never overflows series")
print("=" * 60)

L, H, T = 8, 6, 30
mask    = torch.ones(2, T, 3)
_, block_len = heterogeneous_sampler(mask, L, fcd_samples=1, horizon=H)
new_max_start = T - block_len

overflow_found = False
for _ in range(100):
    ws, _ = heterogeneous_sampler(mask, L, fcd_samples=1, horizon=H)
    if not (ws <= new_max_start).all().item():
        overflow_found = True
        print(f"  overflow: ws={ws.tolist()}, max={new_max_start}")
        break

if not overflow_found:
    print("\n✓ TEST PASSED")

In [ ]:
# %%
import tempfile
import numpy as np
import pandas as pd
import torch
from types import SimpleNamespace
from dataloaders._forking_sequences import fork_sequences
from dataloaders._ts_dataloader import DataLoaderFactory


def make_df(n_series=3, n_steps=500, n_hist=0, seed=42):
    rng   = np.random.default_rng(seed)
    times = pd.date_range("2020-01-01", periods=n_steps, freq="5min")
    rows  = []
    for s in range(n_series):
        y    = rng.standard_normal(n_steps).astype(np.float32)
        hist = {f"hist_{i}": rng.standard_normal(n_steps).astype(np.float32)
                for i in range(n_hist)}
        rows.append(pd.DataFrame({
            "unique_id": f"series_{s}", "ds": times, "y": y,
            "available_mask": np.ones(n_steps, dtype=np.float32), **hist,
        }))
    return pd.concat(rows, ignore_index=True)


def make_mcfg(horizon=6, context_length=64, fcd_samples=4, batch_size=2,
              mixing_strategy="concat", checkpoint_dir="/tmp/ckpts"):
    return SimpleNamespace(
        horizon                 = horizon,
        context_length          = context_length,
        fcd_samples             = fcd_samples,
        batch_size              = batch_size,
        valid_batch_size        = batch_size,
        max_steps               = 20,
        val_check_interval      = 10,
        learning_rate           = 1e-3,
        gradient_clip_val       = 1.0,
        early_stopping_patience = 9999,
        monitor_metric          = "loss",
        monitor_mode            = "min",
        mixing_strategy         = mixing_strategy,
        val_strategy            = "exhaustive",
        drop_last               = False,
        batch_mode              = "full_series",
        checkpoint_dir          = checkpoint_dir,
        checkpoint_step         = 99999,
        num_workers             = 0,
        loss                    = "mae",
        seed                    = 42,
        stat_exog_cols          = [],
    )


def make_entry(path, name="ds", horizon=6, val_size=50, test_size=50,
               weight=1.0, hist_exog_cols=None, futr_exog_cols=None,
               stat_exog_cols=None, per_series_split=False,
               use_context_head=True, sharded_dir=None, multivariate=False):
    return SimpleNamespace(
        path             = path,
        name             = name,
        horizon          = horizon,
        val_size         = val_size,
        test_size        = test_size,
        weight           = weight,
        hist_exog_cols   = hist_exog_cols or [],
        futr_exog_cols   = futr_exog_cols or [],
        stat_exog_cols   = stat_exog_cols or [],
        per_series_split = per_series_split,
        use_context_head = use_context_head,
        sharded_dir      = sharded_dir,
        multivariate     = multivariate,
    )


def make_dcfg(train_entries, val_entries=None, test_entries=None):
    return SimpleNamespace(
        train      = train_entries,
        validation = val_entries  or train_entries,
        test       = test_entries or train_entries,
    )

def check(name, condition, msg=""):
    icon = "OK" if condition else "XX"
    note = "PASS" if condition else f"FAIL  {msg}"
    print(f"  [{icon}] {note}  —  {name}")

def print_masks(outsample_mask, insample_mask, L, H, t_offset=0, max_windows=None):
    """
    Print insample and outsample masks side by side for series 0, channel 0.
    Each row: [context mask (L) | horizon mask (H)]
    """
    n_fcds   = outsample_mask.shape[1]
    enc_size = insample_mask.shape[1]
    n        = min(n_fcds, max_windows) if max_windows else n_fcds
    print(f"\n  (showing {n}/{n_fcds} windows,  L={L}  H={H})")
    print(f"  {'w':>4}  t_range         insample(L={L})  outsample(H={H})")
    print(f"  {''  :>4}  {''  :14}  {'-'*L}  {'-'*H}")
    for w in range(n):
        t_start  = w + t_offset
        t_end    = w + L + H - 1 + t_offset
        in_vals  = insample_mask[0, w : w + L, 0].tolist() if w + L <= enc_size else []
        in_str   = "".join(str(int(v)) for v in in_vals).ljust(L)
        out_vals = outsample_mask[0, w, :, 0].tolist()
        out_str  = "".join(str(int(v)) for v in out_vals)
        print(f"  {w:>4}  t=[{t_start:3d},{t_end:3d}]  {in_str}  {out_str}")

print("helpers OK")

# %%
# ── Setup ─────────────────────────────────────────────────────────────────────

L, H      = 6, 3
val_size  = 9
test_size = 9

with tempfile.TemporaryDirectory() as tmp:
    csv = f"{tmp}/data.csv"
    make_df(n_series=1, n_steps=30).to_csv(csv, index=False)

    mcfg    = make_mcfg(context_length=L, horizon=H)
    entry   = make_entry(csv, name="ds", horizon=H,
                         val_size=val_size, test_size=test_size,
                         use_context_head=True)
    dcfg    = make_dcfg([entry])
    factory = DataLoaderFactory(mcfg, dcfg)

    train_loader = factory.train_dataloader()
    val_loader   = factory.val_dataloaders()["val"]
    test_loader  = factory.test_dataloaders()["test"]

    n_steps  = 30   # must match make_df(n_steps=...)
    T_train  = n_steps - val_size - test_size   # 12
    T_val    = val_size                          # 9
    T_test   = test_size                         # 9

    # absolute start of each dataset in the full series
    # val/test prepend L ctx rows, so their series starts L before their split
    t_offset_train = 0
    t_offset_val   = T_train - L     # ctx starts at last L rows of train
    t_offset_test  = T_train + T_val - L

    # ── Train ─────────────────────────────────────────────────────────────────
    # Layout: [train=1 | H-1 ext=0]
    # All windows fully inside train → mask=1
    # Last H-1 windows overlap ext → partly masked

    print("\n" + "="*60)
    print(f"TRAIN  layout: [train=1 | ext=0]  (ext={H-1} rows)")
    print("="*60)
    for batch in train_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]   # [B, n_fcds, H, C]
        avail          = batch["available_mask"]  # [B, S, C]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_train)

        # outsample_mask[b,w,h,c] must equal avail[b, w+L+h, c]
        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        check("outsample_mask == available_mask at every (w,h)", len(bad) == 0,
              f"mismatches at {bad[:5]}")

        # last H-1 windows should be partially masked (overlap ext)
        n_fcds = outsample_mask.shape[1]
        boundary_partly_masked = any(
            (outsample_mask[0, w, :, 0] == 0).any().item()
            for w in range(max(0, n_fcds - (H - 1)), n_fcds)
        )
        check("last H-1 windows are partially masked (ext rows)", boundary_partly_masked)
        break

    # ── Val ───────────────────────────────────────────────────────────────────
    # Layout: [ctx=0 | val=1 | H-1 ext=0]
    # Windows whose horizons are fully inside val → all mask=1
    # Windows overlapping ctx or ext → partly/fully masked

    print("\n" + "="*60)
    print(f"VAL    layout: [ctx=0 | val=1 | ext=0]  (ctx={L}, ext={H-1} rows)")
    print("="*60)
    for batch in val_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]
        avail          = batch["available_mask"]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_val)

        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        check("outsample_mask == available_mask at every (w,h)", len(bad) == 0,
              f"mismatches at {bad[:5]}")

        n_fcds = outsample_mask.shape[1]
        # windows fully inside val: h_start>=T_ctx and h_end<T_ctx+val_size
        # T_ctx=L (ctx rows), so h_start = w+L >= L → w >= 0 always
        # h_end = w+L+H-1 < L+val_size → w < val_size-H+1
        fully_unmasked = [w for w in range(n_fcds)
                          if (outsample_mask[0, w, :, 0] == 1).all().item()]
        expected_fully_unmasked = val_size - H + 1
        check(f"windows fully inside val are fully unmasked (expected {expected_fully_unmasked})",
              len(fully_unmasked) == expected_fully_unmasked,
              f"got {len(fully_unmasked)}")

        # last H-1 windows should be partially masked (overlap ext)
        boundary_partly_masked = any(
            (outsample_mask[0, w, :, 0] == 0).any().item()
            for w in range(max(0, n_fcds - (H - 1)), n_fcds)
        )
        check("last H-1 windows are partially masked (ext rows)", boundary_partly_masked)
        break

    # ── Test ──────────────────────────────────────────────────────────────────
    # Layout: [ctx=0 | test=1]
    # No extension — windows fully inside test → all mask=1
    # Early windows overlapping ctx → partly masked

    print("\n" + "="*60)
    print(f"TEST   layout: [ctx=0 | test=1]  (ctx={L}, no ext)")
    print("="*60)
    for batch in test_loader:
        out            = fork_sequences(batch, context_length=L, fcd_samples=-1, horizon=H)
        outsample_mask = out["outsample_mask"]
        avail          = batch["available_mask"]
        S              = avail.shape[1]

        print_masks(outsample_mask, out["available_mask"], L, H, t_offset=t_offset_test)

        bad = [(w, h) for w in range(outsample_mask.shape[1])
                      for h in range(H)
                      if outsample_mask[0, w, h, 0].item() != avail[0, min(w+L+h, S-1), 0].item()]
        check("outsample_mask == available_mask at every (w,h)", len(bad) == 0,
              f"mismatches at {bad[:5]}")

        n_fcds = outsample_mask.shape[1]
        fully_unmasked = [w for w in range(n_fcds)
                          if (outsample_mask[0, w, :, 0] == 1).all().item()]
        expected_fully_unmasked = test_size - H + 1
        check(f"windows fully inside test are fully unmasked (expected {expected_fully_unmasked})",
              len(fully_unmasked) == expected_fully_unmasked,
              f"got {len(fully_unmasked)}")

        # no extension rows → last window should be fully unmasked
        check("last window is fully unmasked (no ext for test)",
              (outsample_mask[0, -1, :, 0] == 1).all().item())
        break